# TelAgentBench-ID Unified Benchmark Evaluator and Publication Figures

Evaluates the 12 model configurations reported in Table 7 of the manuscript on the 156-item held-out `TelAgentBench-ID` evaluation suite across 23 BSS APIs:

- **Single-Turn Tool Execution** (`telco_simple.json`): 55 items
- **Single-Turn Ambiguous / Abstention** (`telco_live_irrelevance.json` subset): 55 items
- **Multi-Turn Stateless Dialogues** (`telco_multi_turn_stateless_base.json`): 46 dialogues

Supports Regime A (Zero-Shot schema invocation) and Regime B (3-Shot Prompting using the three Indonesian BSS exemplars), computes strict AST, Macro-F1, format validity, abstention accuracy, BLEU-4, ROUGE-L, and inference latency/throughput, and reproduces Figures 2-4 from the standardized results summary.


## 1. Environment Setup and Credentials


In [ ]:
!pip install -q -U peft accelerate bitsandbytes scikit-learn pandas matplotlib huggingface_hub openai anthropic nltk rouge-score tabulate


In [ ]:
import ast
import gc
import json
import os
import re
import time
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import torch
from huggingface_hub import snapshot_download
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from openai import OpenAI
from peft import PeftModel
from rouge_score import rouge_scorer
from sklearn.metrics import f1_score
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)
except Exception:
    pass

smoothie = SmoothingFunction().method4
rouge_evaluator = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

# Read optional credentials from environment variables (.env).
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip() or None
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip()
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL", "http://localhost:8000/v1").strip()
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip()

client_openrouter = (
    OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")
    if OPENROUTER_API_KEY and not OPENROUTER_API_KEY.startswith("sk-or-v1-your")
    else None
)
client_vllm = (
    OpenAI(api_key=OPENAI_API_KEY or "EMPTY", base_url=OPENAI_BASE_URL)
    if OPENAI_BASE_URL
    else None
)
client_anthropic = None
if ANTHROPIC_API_KEY and not ANTHROPIC_API_KEY.startswith("sk-ant-your"):
    try:
        from anthropic import Anthropic
        client_anthropic = Anthropic(api_key=ANTHROPIC_API_KEY)
    except Exception:
        client_anthropic = None

BASE_OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else ("/content" if os.path.exists("/content") else "../results")
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
print(f"CUDA available: {torch.cuda.is_available()} | Output directory: {BASE_OUTPUT_DIR}")


## 2. Load 156-Item Held-Out Test Suite (`Rislantrs/TelAgentBench-ID`)


In [ ]:
def resolve_telagentbench_dir() -> str:
    candidates = [
        "../../2_HuggingFace_Dataset_TelAgentBench-ID",
        "../2_HuggingFace_Dataset_TelAgentBench-ID",
        "/kaggle/input/telagentbench-id",
        "/content/TelAgentBench-ID",
        "TelAgentBench-ID",
    ]
    for cand in candidates:
        if os.path.exists(os.path.join(cand, "TelAgent_Action")):
            return cand
    print("Local benchmark folder not found; downloading Rislantrs/TelAgentBench-ID from Hugging Face Hub...")
    return snapshot_download(repo_id="Rislantrs/TelAgentBench-ID", repo_type="dataset", token=HF_TOKEN)


DATA_DIR = resolve_telagentbench_dir()
ACTION_DIR = os.path.join(DATA_DIR, "TelAgent_Action")

TOOLS_FILE = os.path.join(ACTION_DIR, "250918_telcoFC_tools.json")
SIMPLE_FILE = os.path.join(ACTION_DIR, "telco_simple.json")
SIMPLE_ANS = os.path.join(ACTION_DIR, "possible_answer", "telco_simple.json")
IRREL_FILE = os.path.join(ACTION_DIR, "telco_live_irrelevance.json")
MULTI_FILE = os.path.join(ACTION_DIR, "telco_multi_turn_stateless_base.json")
MULTI_ANS = os.path.join(ACTION_DIR, "possible_answer", "telco_multi_turn_stateless_base.json")


def _load_jsonl(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


TELCO_TOOLS = _load_jsonl(TOOLS_FILE)
FORMATTED_TOOLS = [
    t if t.get("type") == "function" else {"type": "function", "function": t}
    for t in TELCO_TOOLS
]

# 1. Single-Turn Tool Execution (55 items)
simple_instances = []
for item in _load_jsonl(SIMPLE_FILE):
    item["is_abstention"] = False
    simple_instances.append(item)

simple_answers: Dict[str, List[Dict[str, Any]]] = {}
for item in _load_jsonl(SIMPLE_ANS):
    calls = []
    for gt in item.get("ground_truth", []):
        if isinstance(gt, dict):
            if "name" in gt:
                calls.append(gt)
            else:
                for fname, fargs in gt.items():
                    calls.append({"name": fname, "arguments": fargs})
    simple_answers[item["id"]] = calls

# 2. Single-Turn Ambiguous / Abstention Subset (first 55 balanced items)
for item in _load_jsonl(IRREL_FILE)[:55]:
    q_content = item.get("question", "")
    user_msg = ""
    if isinstance(q_content, list):
        for turn in q_content:
            msgs = turn if isinstance(turn, list) else [turn]
            for m in msgs:
                if isinstance(m, dict) and m.get("role") == "user":
                    user_msg = m.get("content", "")
    simple_instances.append({
        "id": item["id"],
        "question": user_msg if user_msg else str(q_content),
        "is_abstention": True,
    })
    simple_answers[item["id"]] = []

# 3. Multi-Turn Stateless Dialogues (46 dialogues)
multi_instances = _load_jsonl(MULTI_FILE)
multi_answers: Dict[str, List[List[Dict[str, Any]]]] = {}
for item in _load_jsonl(MULTI_ANS):
    all_turns = []
    for turn_list in item.get("ground_truth", []):
        turn_calls = []
        for call_str in turn_list:
            m = re.match(r"^([A-Za-z0-9_]+__[\w\-]+)\((.*)\)$", str(call_str).strip())
            if m:
                fname, args_raw = m.group(1), m.group(2)
                arg_dict = {k: v for k, v in re.findall(r"(\w+)\s*=\s*'([^']*)'", args_raw)}
                for k, v in re.findall(r"(\w+)\s*=\s*(\d+)(?:\s*[,)]|\s*$)", args_raw):
                    if k not in arg_dict:
                        arg_dict[k] = int(v)
                turn_calls.append({"name": fname, "arguments": arg_dict})
        all_turns.append(turn_calls)
    multi_answers[item["id"]] = all_turns

print(
    f"Loaded {len(FORMATTED_TOOLS)} BSS tools | "
    f"Single-Turn: {len(simple_instances)} (55 exec + 55 abstention) | "
    f"Multi-Turn: {len(multi_instances)} dialogues"
)


## 3. Prompt Regimes (A and B) and 12-Model Registry


In [ ]:
SYSTEM_PROMPT_REGIME_A = (
    "Anda adalah asisten AI resmi Customer Care TelkomNusa. "
    "Panggil tool yang sesuai jika diperlukan."
)

SYSTEM_PROMPT_REGIME_B = """Anda adalah agen API resmi Customer Care TelkomNusa Indonesia.

ATURAN EKSEKUSI:
1. Jika pelanggan meminta transaksi (cek pulsa, kuota, tagihan, bayar tunggakan, aktivasi/pembatalan paket, roaming, kupon extra kuota, info profil), panggil tool API yang sesuai dari daftar tools.
2. Format pemanggilan tool menggunakan tag XML:
   <tool_call>{"name": "NAMA_API", "arguments": {"param": "value"}}</tool_call>
3. Jangan menjawab dengan instruksi manual jika pertanyaan membutuhkan eksekusi tool.
4. Jika pertanyaan berada di luar cakupan 23 API BSS atau tidak membutuhkan tool, jawab secara wajar tanpa memanggil tool.
5. Ekstrak parameter (nomor telepon, tanggal, ID produk, kode negara) langsung dari konteks percakapan."""

REGIME_B_FEW_SHOT_EXAMPLES = [
    {"role": "user", "content": "Mohon periksa tunggakan tagihan untuk nomor 0109998888."},
    {"role": "assistant", "content": '<tool_call>{"name": "GET__rate_unpaid-bills", "arguments": {"svcMgmtNum": "0109998888"}}</tool_call>'},
    {"role": "user", "content": "Tolong cek sisa kuota internet saya, nomor 0101234567."},
    {"role": "assistant", "content": '<tool_call>{"name": "GET__rate_current-data-limits", "arguments": {"svcMgmtNum": "0101234567"}}</tool_call>'},
    {"role": "user", "content": "Tampilkan tagihan anggota keluarga nomor 01055556666 yang terdaftar di nomor utama 01077778888."},
    {"role": "assistant", "content": '<tool_call>{"name": "GET__rate_family_bills", "arguments": {"familySvcMgmtNum": "01055556666", "svcMgmtNum": "01077778888"}}</tool_call>'},
]

# Standardized 12 model configurations matching Table 7 in the manuscript.
# Toggle "active": True for the models you want to run in the current compute session.
MODEL_REGISTRY = {
    "TelkomNusa-LLM-7B": {
        "platform": "local_lora",
        "base_model": "Qwen/Qwen2.5-7B-Instruct",
        "adapter": "Rislantrs/telecom-troubleshooter-qwen2.5-7b-balanced-lora",
        "category": "Proposed LoRA SLM",
        "regime": "Regime A",
        "active": True,
    },
    "TelkomNusa-SLM-3B": {
        "platform": "local_lora",
        "base_model": "Qwen/Qwen2.5-3B-Instruct",
        "adapter": "Rislantrs/telecom-troubleshooter-qwen2.5-3b-balanced-lora",
        "category": "Proposed LoRA SLM",
        "regime": "Regime A",
        "active": True,
    },
    "Qwen2.5-7B-Instruct (Untuned)": {
        "platform": "local_hf",
        "model_id": "Qwen/Qwen2.5-7B-Instruct",
        "category": "Open-Weight Baseline",
        "regime": "Regime A",
        "active": False,
    },
    "Qwen2.5-3B-Instruct (Untuned)": {
        "platform": "local_hf",
        "model_id": "Qwen/Qwen2.5-3B-Instruct",
        "category": "Open-Weight Baseline",
        "regime": "Regime A",
        "active": False,
    },
    "Llama-3.2-3B-Instruct": {
        "platform": "local_hf",
        "model_id": "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
        "category": "Open-Weight Baseline",
        "regime": "Regime B",
        "active": False,
    },
    "Phi-4-mini": {
        "platform": "local_hf",
        "model_id": "microsoft/Phi-4-mini-instruct",
        "category": "Open-Weight Baseline",
        "regime": "Regime B",
        "active": False,
    },
    "Gemma4-31B": {
        "platform": "api_vllm",
        "model_id": "google/gemma-4-31b-it",
        "category": "Mid-to-Large Open",
        "regime": "Regime A",
        "active": False,
    },
    "GPT-OSS-20B": {
        "platform": "api_vllm",
        "model_id": "openai/gpt-oss-20b",
        "category": "Mid-to-Large Open",
        "regime": "Regime A",
        "active": False,
    },
    "GPT-OSS-120B": {
        "platform": "api_vllm",
        "model_id": "openai/gpt-oss-120b",
        "category": "Mid-to-Large Open",
        "regime": "Regime A",
        "active": False,
    },
    "Nemotron-30B": {
        "platform": "api_vllm",
        "model_id": "nvidia/nemotron-30b",
        "category": "Mid-to-Large Open",
        "regime": "Regime A",
        "active": False,
    },
    "MiniMax-M3": {
        "platform": "api_openrouter",
        "model_id": "minimax/minimax-m3",
        "category": "Proprietary Cloud",
        "regime": "Regime B",
        "active": False,
    },
    "Claude-Opus-5": {
        "platform": "api_anthropic",
        "model_id": "claude-opus-5",
        "category": "Proprietary Cloud",
        "regime": "Regime B",
        "active": False,
    },
}

print(f"Registered {len(MODEL_REGISTRY)} Table-7 configurations.")


## 4. Four-Tier Cascade Parser and Deterministic AST Evaluator


In [ ]:
def extract_tool_calls(response_text: str) -> List[Dict[str, Any]]:
    """Parse tool invocations across <tool_call> XML, Markdown JSON, Pythonic calls, and bare JSON."""
    calls: List[Dict[str, Any]] = []
    if not response_text:
        return calls
    resp = response_text.replace("\\n", "\n")

    # Tier 1: <tool_call>...</tool_call>
    for m in re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", resp, re.DOTALL):
        try:
            parsed = json.loads(m.strip())
            if isinstance(parsed, dict) and "name" in parsed:
                args = parsed.get("arguments", {})
                if isinstance(args, str):
                    try:
                        args = json.loads(args)
                    except Exception:
                        args = {}
                calls.append({"name": parsed["name"], "arguments": args if isinstance(args, dict) else {}})
        except Exception:
            pass
    if calls:
        return calls

    # Tier 2: Fenced Markdown JSON block
    for jb in re.findall(r"```(?:json)?\s*([\[\{].*?[\]\}])\s*```", resp, re.DOTALL):
        try:
            parsed = json.loads(jb.strip())
            for it in (parsed if isinstance(parsed, list) else [parsed]):
                if isinstance(it, dict) and "name" in it:
                    args = it.get("arguments", {})
                    if isinstance(args, str):
                        try:
                            args = json.loads(args)
                        except Exception:
                            args = {}
                    calls.append({"name": it["name"], "arguments": args if isinstance(args, dict) else {}})
        except Exception:
            pass
    if calls:
        return calls

    # Tier 3: Pythonic function-call syntax NAME__endpoint(key='val')
    for fname, args_str in re.findall(r"([A-Za-z0-9_]+__[\w\-]+)\(([^)]*)\)", resp):
        arg_dict: Dict[str, Any] = {}
        for km, vm in re.findall(r"(\w+)\s*=\s*'([^']*)'", args_str):
            arg_dict[km] = vm
        for km, vm in re.findall(r"(\w+)\s*=\s*(\d+)(?:\s*[,)]|\s*$)", args_str):
            if km not in arg_dict:
                arg_dict[km] = int(vm)
        calls.append({"name": fname, "arguments": arg_dict})
    if calls:
        return calls

    # Tier 4: Bare JSON fallback
    for fname, args_raw in re.findall(r'\{"name"\s*:\s*"([^"]+)"\s*,\s*"arguments"\s*:\s*(\{[^}]+\})\}', resp):
        try:
            arg_dict = json.loads(args_raw)
            calls.append({"name": fname, "arguments": arg_dict if isinstance(arg_dict, dict) else {}})
        except Exception:
            calls.append({"name": fname, "arguments": {}})
    return calls


def normalize_tool_name(name: str) -> str:
    return str(name or "").strip().lower().replace("-", "_")


def compare_arguments(pred_args: Dict[str, Any], gold_args: Dict[str, Any]) -> bool:
    if not gold_args:
        return True
    if not pred_args:
        return False
    for k, g_val in gold_args.items():
        if k not in pred_args:
            if isinstance(g_val, list) and (None in g_val or "" in g_val):
                continue
            return False
        p_val_str = str(pred_args[k]).strip().lower()
        if isinstance(g_val, list):
            allowed = [str(x).strip().lower() for x in g_val if x is not None]
            if allowed and p_val_str not in allowed:
                return False
        elif p_val_str != str(g_val).strip().lower():
            return False
    return True


def score_single_turn_instance(
    pred_calls: List[Dict[str, Any]],
    gold_calls: List[Dict[str, Any]],
    is_abstention_sample: bool = False,
) -> Tuple[float, float, float, float, float, str]:
    if not gold_calls or is_abstention_sample:
        if not pred_calls:
            return 1.0, 1.0, 1.0, 1.0, 1.0, "Correct (Abstention)"
        return 0.0, 0.0, 0.0, 1.0, 0.0, "Spurious Tool Call (Failed Abstention)"

    if not pred_calls:
        return 0.0, 0.0, 0.0, 0.0, 0.0, "Format / No Tool Emitted"

    if normalize_tool_name(pred_calls[0].get("name", "")) != normalize_tool_name(gold_calls[0].get("name", "")):
        return (
            0.0,
            0.0,
            0.0,
            1.0,
            0.0,
            f"Wrong Tool (Pred: {pred_calls[0].get('name')} vs Gold: {gold_calls[0].get('name')})",
        )

    if compare_arguments(pred_calls[0].get("arguments", {}), gold_calls[0].get("arguments", {})):
        return 1.0, 1.0, 1.0, 1.0, 1.0, "Correct"
    return 1.0, 0.0, 0.0, 1.0, 1.0, "Correct Tool, Wrong Arguments"


def score_multi_turn_instance(
    pred_turns: List[List[Dict[str, Any]]],
    gold_turns: List[List[Dict[str, Any]]],
) -> Tuple[float, float, float, float, str]:
    num_turns = len(gold_turns)
    if num_turns == 0:
        return 1.0, 1.0, 1.0, 1.0, "Correct (No Turns)"

    tool_scores, arg_scores, strict_scores = [], [], []
    total_calls_pred = 0

    for t in range(num_turns):
        gold_turn_calls = gold_turns[t]
        pred_turn_calls = pred_turns[t] if t < len(pred_turns) else []
        total_calls_pred += len(pred_turn_calls)

        if not gold_turn_calls:
            score = 1.0 if not pred_turn_calls else 0.0
            tool_scores.append(score)
            arg_scores.append(score)
            strict_scores.append(score)
            continue

        if not pred_turn_calls:
            tool_scores.append(0.0)
            arg_scores.append(0.0)
            strict_scores.append(0.0)
            continue

        t_matches, s_matches = 0, 0
        for g_call in gold_turn_calls:
            matched_tool = False
            matched_strict = False
            for p_call in pred_turn_calls:
                if normalize_tool_name(p_call.get("name", "")) == normalize_tool_name(g_call.get("name", "")):
                    matched_tool = True
                    if compare_arguments(p_call.get("arguments", {}), g_call.get("arguments", {})):
                        matched_strict = True
                        break
            t_matches += int(matched_tool)
            s_matches += int(matched_strict)

        t_acc = t_matches / len(gold_turn_calls)
        s_ast = s_matches / len(gold_turn_calls)
        tool_scores.append(t_acc)
        arg_scores.append(s_ast if t_matches > 0 else 0.0)
        strict_scores.append(s_ast)

    return (
        sum(tool_scores) / num_turns,
        sum(arg_scores) / num_turns,
        sum(strict_scores) / num_turns,
        1.0 if total_calls_pred > 0 else 0.0,
        f"Per-Turn Evaluated ({num_turns} Turns)",
    )


def compute_nlg_metrics(pred_text: str, gold_calls: Any) -> Tuple[float, float]:
    if not pred_text or not gold_calls:
        return 0.0, 0.0
    ref_parts = []
    for gc in gold_calls:
        if isinstance(gc, dict):
            ref_parts.append(json.dumps(gc, sort_keys=True, ensure_ascii=False))
        elif isinstance(gc, list):
            for g in gc:
                if isinstance(g, dict):
                    ref_parts.append(json.dumps(g, sort_keys=True, ensure_ascii=False))
    ref_text = " ".join(ref_parts).strip()
    if not ref_text:
        return 0.0, 0.0
    try:
        bleu = sentence_bleu([ref_text.lower().split()], pred_text.lower().split(), smoothing_function=smoothie)
    except Exception:
        bleu = 0.0
    try:
        rouge_l = rouge_evaluator.score(ref_text, pred_text)["rougeL"].fmeasure
    except Exception:
        rouge_l = 0.0
    return round(float(bleu) * 100.0, 2), round(float(rouge_l) * 100.0, 2)


## 5. Multi-Platform Inference and Evaluation Runner


In [ ]:
CURRENT_MODEL = None
CURRENT_TOKENIZER = None
CURRENT_MODEL_KEY = None


def unload_local_model() -> None:
    global CURRENT_MODEL, CURRENT_TOKENIZER, CURRENT_MODEL_KEY
    if CURRENT_MODEL is not None:
        del CURRENT_MODEL
        CURRENT_MODEL = None
    if CURRENT_TOKENIZER is not None:
        del CURRENT_TOKENIZER
        CURRENT_TOKENIZER = None
    CURRENT_MODEL_KEY = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_local_model(cfg: Dict[str, Any]) -> Tuple[Any, Any]:
    global CURRENT_MODEL, CURRENT_TOKENIZER, CURRENT_MODEL_KEY
    key = cfg.get("adapter") or cfg.get("model_id")
    if CURRENT_MODEL_KEY == key and CURRENT_MODEL is not None:
        return CURRENT_MODEL, CURRENT_TOKENIZER

    unload_local_model()
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    base_id = cfg["base_model"] if cfg["platform"] == "local_lora" else cfg["model_id"]
    tokenizer = AutoTokenizer.from_pretrained(base_id, token=HF_TOKEN, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        base_id,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        token=HF_TOKEN,
        trust_remote_code=True,
    )
    if cfg["platform"] == "local_lora":
        model = PeftModel.from_pretrained(model, cfg["adapter"], token=HF_TOKEN)
    model.eval()

    CURRENT_MODEL, CURRENT_TOKENIZER, CURRENT_MODEL_KEY = model, tokenizer, key
    return model, tokenizer


def run_chat_inference(messages: List[Dict[str, str]], cfg: Dict[str, Any]) -> Tuple[str, float, int]:
    platform = cfg["platform"]

    if platform in ("local_lora", "local_hf"):
        model, tokenizer = load_local_model(cfg)
        prompt_text = tokenizer.apply_chat_template(
            messages, tools=FORMATTED_TOOLS, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")
        eos_ids = [tokenizer.eos_token_id]
        for tok in ("<|im_end|>", "<|endoftext|>"):
            tid = tokenizer.convert_tokens_to_ids(tok)
            if isinstance(tid, int) and tid > 0:
                eos_ids.append(tid)
        t0 = time.time()
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=list(set(eos_ids)),
            )
        lat = time.time() - t0
        gen_tokens = output_ids[0][inputs["input_ids"].shape[1] :]
        resp = tokenizer.decode(gen_tokens, skip_special_tokens=False).strip()
        for stop_tok in ("<|im_end|>", "<|endoftext|>", "</s>"):
            resp = resp.replace(stop_tok, "").strip()
        return resp, lat, int(len(gen_tokens))

    if platform in ("api_vllm", "api_openrouter"):
        client = client_vllm if platform == "api_vllm" else client_openrouter
        if client is None:
            raise RuntimeError(f"Client for {platform} is not configured.")
        t0 = time.time()
        res = client.chat.completions.create(
            model=cfg["model_id"],
            messages=messages,
            tools=FORMATTED_TOOLS,
            temperature=0.0,
        )
        lat = time.time() - t0
        msg = res.choices[0].message
        if msg.tool_calls:
            resp = json.dumps([
                {"name": tc.function.name, "arguments": json.loads(tc.function.arguments)}
                for tc in msg.tool_calls
            ])
        else:
            resp = msg.content or ""
        n_tok = res.usage.completion_tokens if getattr(res, "usage", None) else 0
        return resp, lat, int(n_tok)

    if platform == "api_anthropic":
        if client_anthropic is None:
            raise RuntimeError("ANTHROPIC_API_KEY is not configured.")
        sys_msg = "\n".join(m["content"] for m in messages if m["role"] == "system")
        conv = [m for m in messages if m["role"] != "system"]
        t0 = time.time()
        res = client_anthropic.messages.create(
            model=cfg["model_id"],
            system=sys_msg,
            messages=conv,
            max_tokens=512,
            temperature=0.0,
        )
        lat = time.time() - t0
        resp = "".join(block.text for block in res.content if hasattr(block, "text"))
        return resp, lat, int(getattr(res.usage, "output_tokens", 0))

    raise ValueError(f"Unsupported platform: {platform}")


def build_initial_messages(cfg: Dict[str, Any], raw_sys: str = "") -> List[Dict[str, str]]:
    svc_match = re.search(r'"svc_num"\s*:\s*"([^"]+)"', raw_sys)
    active_svc_num = svc_match.group(1) if svc_match else ""

    if cfg["regime"] == "Regime B":
        sys_content = SYSTEM_PROMPT_REGIME_B
        if active_svc_num:
            sys_content += f"\n[INFORMASI PELANGGAN AKTIF]: svcMgmtNum = {active_svc_num}"
        return [{"role": "system", "content": sys_content}, *REGIME_B_FEW_SHOT_EXAMPLES]

    sys_content = SYSTEM_PROMPT_REGIME_A
    if active_svc_num:
        sys_content += (
            f"\n[INFORMASI PELANGGAN AKTIF]: Nomor layanan pelanggan yang sedang dilayani "
            f"adalah {active_svc_num}. Gunakan nomor ini sebagai parameter svcMgmtNum jika diperlukan."
        )
    return [{"role": "system", "content": sys_content}]


def evaluate_configuration(model_name: str, cfg: Dict[str, Any]) -> Dict[str, Any]:
    st_records, mt_records = [], []
    all_bleu, all_rouge = [], []
    total_tokens, total_lat = 0, 0.0

    for inst in tqdm(simple_instances, desc=f"{model_name} [ST]"):
        q_id = inst["id"]
        q_content = inst["question"]
        gold = simple_answers.get(q_id, [])
        is_abst = inst.get("is_abstention", False)

        messages = build_initial_messages(cfg)
        if isinstance(q_content, list):
            for item in q_content:
                messages.extend(item if isinstance(item, list) else [item])
        else:
            messages.append({"role": "user", "content": str(q_content)})

        resp, lat, n_tok = run_chat_inference(messages, cfg)
        total_tokens += n_tok
        total_lat += lat

        p_calls = extract_tool_calls(resp)
        t_acc, a_m, s_ast, f_val, abst_acc, cat = score_single_turn_instance(
            p_calls, gold, is_abstention_sample=is_abst
        )
        bleu_v, rouge_v = compute_nlg_metrics(resp, gold)
        all_bleu.append(bleu_v)
        all_rouge.append(rouge_v)

        st_records.append({
            "id": q_id,
            "type": "Single-Turn",
            "is_abstention": is_abst,
            "raw_response": resp,
            "gold_tool": gold[0]["name"] if gold else "NONE",
            "pred_tool": p_calls[0]["name"] if p_calls else "NONE",
            "tool_accuracy": t_acc,
            "arg_match": a_m,
            "strict_ast": s_ast,
            "format_valid": f_val,
            "abstention_acc": abst_acc,
            "category": cat,
            "latency_s": lat,
            "bleu4": bleu_v,
            "rouge_l": rouge_v,
        })

    for inst in tqdm(multi_instances, desc=f"{model_name} [MT]"):
        q_id = inst["id"]
        q_content = inst["question"]
        gold_t = multi_answers.get(q_id, [])

        raw_sys = ""
        if isinstance(q_content, list) and q_content and isinstance(q_content[0], list):
            for tm in q_content[0]:
                if tm.get("role") == "system":
                    raw_sys = tm.get("content", "")
                    break

        history = build_initial_messages(cfg, raw_sys=raw_sys)
        pred_turns, turn_lats, turn_resps = [], [], []

        if isinstance(q_content, list) and q_content and isinstance(q_content[0], list):
            for turn_msgs in q_content:
                for tm in turn_msgs:
                    if tm.get("role") != "system":
                        history.append(tm)
                t_resp, t_lat, n_tok = run_chat_inference(history, cfg)
                total_tokens += n_tok
                total_lat += t_lat
                turn_lats.append(t_lat)
                turn_resps.append(t_resp)
                history.append({"role": "assistant", "content": t_resp})
                pred_turns.append(extract_tool_calls(t_resp))

        t_acc, a_m, s_ast, f_val, cat = score_multi_turn_instance(pred_turns, gold_t)
        mt_gold_flat = [g for gt in gold_t for g in (gt if isinstance(gt, list) else [gt])]
        bleu_v, rouge_v = compute_nlg_metrics(" ".join(turn_resps), mt_gold_flat)
        all_bleu.append(bleu_v)
        all_rouge.append(rouge_v)

        mt_records.append({
            "id": q_id,
            "type": "Multi-Turn",
            "is_abstention": False,
            "raw_response": json.dumps(pred_turns),
            "gold_tool": gold_t[0][0]["name"] if (gold_t and gold_t[0]) else "NONE",
            "pred_tool": pred_turns[0][0]["name"] if (pred_turns and pred_turns[0]) else "NONE",
            "tool_accuracy": t_acc,
            "arg_match": a_m,
            "strict_ast": s_ast,
            "format_valid": f_val,
            "abstention_acc": 0.0,
            "category": cat,
            "latency_s": float(np.mean(turn_lats)) if turn_lats else 0.0,
            "bleu4": bleu_v,
            "rouge_l": rouge_v,
        })

    unload_local_model()
    df_all = pd.DataFrame(st_records + mt_records)
    pred_csv = os.path.join(BASE_OUTPUT_DIR, f"{model_name}_predictions.csv")
    df_all.to_csv(pred_csv, index=False)
    print(f"Saved per-item predictions to: {pred_csv}")

    st_tool = df_all[(df_all["type"] == "Single-Turn") & (df_all["is_abstention"] == False)]
    mt_df = df_all[df_all["type"] == "Multi-Turn"]
    st_ast = st_tool["strict_ast"].mean() * 100
    mt_ast = mt_df["strict_ast"].mean() * 100

    return {
        "Model Name": model_name,
        "Category": cfg["category"],
        "Regime": cfg["regime"],
        "ST Tool Acc (%)": round(st_tool["tool_accuracy"].mean() * 100, 2),
        "ST Arg Match (%)": round(st_tool["arg_match"].mean() * 100, 2),
        "ST Strict AST (%)": round(st_ast, 2),
        "MT Strict AST (%)": round(mt_ast, 2),
        "Macro Strict AST (%)": round((st_ast + mt_ast) / 2, 2),
        "Macro-F1 (%)": round(f1_score(df_all["gold_tool"], df_all["pred_tool"], average="macro", zero_division=0) * 100, 2),
        "Format Valid (%)": round(df_all["format_valid"].mean() * 100, 2),
        "Latency (s)": round(df_all["latency_s"].mean(), 2),
        "Throughput (tok/s)": round(total_tokens / max(0.01, total_lat), 2),
        "BLEU-4": round(float(np.mean(all_bleu)), 2),
        "ROUGE-L": round(float(np.mean(all_rouge)), 2),
    }


# Set RUN_LIVE_EVAL = True on a GPU runtime to execute active models above.
RUN_LIVE_EVAL = False
if RUN_LIVE_EVAL:
    run_rows = []
    for name, cfg in MODEL_REGISTRY.items():
        if cfg.get("active", False):
            run_rows.append(evaluate_configuration(name, cfg))
    pd.DataFrame(run_rows).to_csv(os.path.join(BASE_OUTPUT_DIR, "Live_Evaluation_Summary.csv"), index=False)


## 6. Reproduce Manuscript Figures (Figures 2, 3, and 4) from Standardized Results


In [ ]:
# Load the standardized 12-model Table-7 CSV from results/ and run publication plotting script.
import subprocess
import sys

summary_candidates = [
    "../results/Master_Standardized_Benchmark_Summary.csv",
    "results/Master_Standardized_Benchmark_Summary.csv",
]
summary_csv = next((p for p in summary_candidates if os.path.exists(p)), None)
if summary_csv:
    df_table7 = pd.read_csv(summary_csv)
    display(df_table7)

script_candidates = [
    "../scripts/generate_3_publication_figures.py",
    "scripts/generate_3_publication_figures.py",
]
fig_script = next((p for p in script_candidates if os.path.exists(p)), None)
if fig_script:
    subprocess.run([sys.executable, fig_script], check=True)
